# AI Studio Worker - Google Colab

Ce notebook prépare une machine GPU Google Colab pour qu’elle se connecte à ton **AI Studio Hugging Face Space**.

Il fait :

1. cloner le projet AI Studio,
2. installer les dépendances Python,
3. créer un tunnel ngrok,
4. lancer le worker GPU,
5. afficher les logs en direct.

## 1. Configuration

Remplace :

- `SERVER_URL` par l’URL de ton Space Hugging Face,
- `WORKER_TOKEN` par le token worker généré dans l’interface AI Studio.

In [ ]:
SERVER_URL = "https://ton-space.hf.space"
WORKER_TOKEN = "TON_WORKER_TOKEN"
WORKER_PORT = 8765

# Optionnel : si tu as un token ngrok, tu peux le mettre ici.
# Sinon, ngrok peut fonctionner avec une URL temporaire.
NGROK_AUTH_TOKEN = ""

REPO_URL = "https://github.com/NathMen12/AI-Studio.git"
PROJECT_DIR = "/content/AI-Studio"

## 2. Cloner le projet

In [ ]:
import os

if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR

%cd $PROJECT_DIR

## 3. Installer les dépendances GPU

Cette cellule installe les dépendances du worker et de l’entraînement Hugging Face LoRA/PEFT.

In [ ]:
!pip install -q -r worker/requirements.txt pyngrok

## 4. Démarrer ngrok

L’URL affichée ici est celle à enregistrer dans l’interface AI Studio si elle n’y est pas déjà.

In [ ]:
from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(WORKER_PORT, "http").public_url

with open("/content/ai_studio_ngrok_url.txt", "w", encoding="utf-8") as file:
    file.write(public_url)

print("URL ngrok du worker :")
print(public_url)

## 5. Lancer le worker GPU

Le worker va se connecter automatiquement au serveur AI Studio avec :

- ton URL de Space,
- ton worker token,
- l’URL ngrok.

In [ ]:
import subprocess
import threading

command = [
    "python",
    "worker.py",
    "--server-url", SERVER_URL,
    "--worker-token", WORKER_TOKEN,
    "--ngrok-url", public_url,
    "--port", str(WORKER_PORT)
]

print("Commande lancée :")
print(" ".join(command))

worker_process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def read_logs():
    for line in worker_process.stdout:
        print(line, end="")

log_thread = threading.Thread(target=read_logs, daemon=True)
log_thread.start()

print("Worker lancé. Retourne dans l’interface AI Studio pour vérifier que la machine est online.")

## 6. Arrêter le worker et ngrok

À exécuter quand tu as terminé.

In [ ]:
try:
    worker_process.terminate()
    worker_process.wait(timeout=10)
except Exception:
    pass

try:
    ngrok.disconnect()
    ngrok.kill()
except Exception:
    pass

print("Worker et ngrok arrêtés.")